In [ ]:
import os 
from pyspark.sql import SparkSession
from pyspark.sql.functions import column

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,") \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [ ]:
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
table_name = "public.shops"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
shops_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name) \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [ ]:
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
table_name = "public.shop_timezone"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
shop_timezone_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name) \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [ ]:
shops_df.createOrReplaceTempView("shops")
shop_timezone_df.createOrReplaceTempView("shop_timezone")

In [ ]:
shops_df.createOrReplaceTempView("shops")
shop_timezone_df.createOrReplaceTempView("shop_timezone")
spark.sql(
    ''' 
    with cte AS (
      select 
        plant, 
        time_zone,
        count(*) over (partition by plant) as counts_plant
      from shop_timezone
    ),
    tz as (
    select plant, 
          case when time_zone != '' then regexp_extract(time_zone, '[1-9]+', 0)
          else 3
          end as tz_code 
    from cte c
    where c.counts_plant = 1 or c.counts_plant > 1 and c.time_zone != ''
    )
    
    select s.st_id, s.shop_name, tz.tz_code from shops s
    inner join tz on tz.plant=s.st_id
    '''
).show()